# DeltIEt — Battery Degradation Early Warning
## Independent Validation Notebook

**Inventor:** Ubirajara dos Santos Vasconcelos — Feira de Santana, BA, Brazil  
**Patent:** BR 10 2026 012158 4 (INPI Brazil, filed 18 May 2026)  
**Contact:** deltiet.research@gmail.com

---

Click **Runtime → Run All**. The notebook downloads data from Zenodo automatically (~1.3 GB, ~10 min).

**Formula:** DeltIEt(t) = Z_t * |Delta2(S_t)|

- S_t = Shannon entropy of capacity returns, rolling window w=15
- Delta2 = second-order finite difference (entropic acceleration)
- Z_t = adaptive z-score over the preceding h=40 cycles

Alarm fires when DeltIEt crosses a rolling p90 threshold. **Zero look-ahead. Zero training.**


In [ ]:
import requests, zipfile, io, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
print('Libraries ready.')


In [ ]:
# DeltIEt v4 — identical parameters across all datasets
W, H, N_BINS, PTILE = 15, 40, 5, 90

def shannon_entropy(vals, n_bins=N_BINS):
    if len(vals) < 3: return float('nan')
    hist, _ = np.histogram(vals, bins=n_bins)
    t = hist.sum()
    if t == 0: return float('nan')
    p = hist[hist>0] / t
    return -np.sum(p * np.log(p))

def run_deltiet(cap):
    s = cap.astype(float); n = len(s)
    S = np.full(n, np.nan); d2S = np.full(n, np.nan)
    Z = np.full(n, np.nan); DI = np.full(n, np.nan)
    alarm = np.zeros(n, bool)
    ret = np.diff(s) / (np.abs(s[:-1]) + 1e-10)
    for t in range(W, n-1): S[t] = shannon_entropy(ret[t-W:t])
    for t in range(1, n-1):
        if not any(np.isnan([S[t-1],S[t],S[t+1]])): d2S[t]=S[t+1]-2*S[t]+S[t-1]
    for t in range(H, n):
        b = d2S[max(0,t-H):t]; b = b[~np.isnan(b)]
        if len(b)>5:
            mu,sig = np.mean(b),np.std(b)
            if sig>1e-10: Z[t]=(d2S[t]-mu)/sig
    for t in range(n):
        if not (np.isnan(Z[t]) or np.isnan(d2S[t])): DI[t]=Z[t]*abs(d2S[t]) if Z[t]<0 else 0.0
    thr_s = np.full(n, np.nan)
    for t in range(H, n):
        w = DI[max(0,t-H):t]; nw = w[(w<0)&~np.isnan(w)]
        if len(nw)>=3:
            th = np.percentile(nw, 100-PTILE); thr_s[t]=th
            if not np.isnan(DI[t]) and DI[t]<th: alarm[t]=True
    return pd.DataFrame({'cap':s,'DeltIEt':DI,'threshold':thr_s,'alarm':alarm})

def filter_outliers(cap):
    c = cap.copy().astype(float); out=[]
    for i in range(5,len(c)):
        m = np.mean(c[max(0,i-5):i])
        if c[i]<0.5*m: out.append(i); c[i]=np.nan
    if out:
        nans=np.isnan(c); idx=np.arange(len(c))
        c[nans]=np.interp(idx[nans],idx[~nans],c[~nans])
    return c, out

def knee_80(cap): idx=np.where(cap<0.8*cap[0])[0]; return int(idx[0]) if len(idx) else None

def load_csv(f):
    try: df=pd.read_csv(f,low_memory=False)
    except: return None
    df.columns=[c.strip() for c in df.columns]
    cc=next((c for c in df.columns if ('discharge' in c.lower() or 'dchg' in c.lower())
             and any(x in c.lower() for x in ['q','mah','ah','cap'])),None)
    cyc=next((c for c in df.columns if 'cycle' in c.lower()),None)
    if not cc or not cyc: return None
    df[cc]=pd.to_numeric(df[cc],errors='coerce'); df[cyc]=pd.to_numeric(df[cyc],errors='coerce')
    df=df.dropna(subset=[cc,cyc])
    s=df.groupby(cyc)[cc].max(); s=s[s>0]
    return s.values if len(s)>50 else None

def run_dataset(files, label):
    cells={os.path.basename(f).replace('.csv',''): load_csv(f) for f in files}
    cells={k:v for k,v in cells.items() if v is not None}
    print(f'\n{label}: {len(cells)} cells')
    summ=[]; res={}
    for name, cap in cells.items():
        cf,_=filter_outliers(cap); r=run_deltiet(cap); res[name]=(r,cap)
        k=knee_80(cf); ai=np.where(r['alarm'].values)[0]
        fa=None
        if k and len(ai): b=ai[ai<k]; fa=int(b[0]) if len(b) else None
        lead=(k-fa) if (k and fa is not None) else None
        summ.append({'name':name,'knee':k,'fa':fa,'lead':lead,
                     'has_knee':k is not None,'det':lead is not None})
    df=pd.DataFrame(summ); wk=df['has_knee'].sum(); det=df['det'].sum()
    leads=df[df['det']]['lead']
    print(f'  knee<=80%: {wk} | detected: {det} ({100*det/wk:.1f}%)')
    if len(leads): print(f'  lead  median={leads.median():.0f}  mean={leads.mean():.0f}  min={leads.min()}  max={leads.max()}')
    return df, res

print('DeltIEt v4 ready. w=15  h=40  n_bins=5  p90')


In [ ]:
# Download NCA — Dataset 1 (Zenodo 10.5281/zenodo.6405084)
print('Downloading NCA ~580 MB ...')
r=requests.get('https://zenodo.org/records/6405084/files/Dataset_1_NCA_battery.zip?download=1',
               stream=True,timeout=300)
r.raise_for_status()
tot=int(r.headers.get('content-length',0)); chunks=[]; dl=0
for ch in r.iter_content(1024*1024):
    chunks.append(ch); dl+=len(ch)
    if tot: print(f'\r  {dl/1e6:.0f}/{tot/1e6:.0f} MB',end='')
print('\nExtracting...')
zipfile.ZipFile(io.BytesIO(b''.join(chunks))).extractall('/content/nca/')
print('Done.')


In [ ]:
# Download NCM — Dataset 2
print('Downloading NCM ~737 MB ...')
r=requests.get('https://zenodo.org/records/6405084/files/Dataset_2_NCM_battery.zip?download=1',
               stream=True,timeout=300)
r.raise_for_status()
tot=int(r.headers.get('content-length',0)); chunks=[]; dl=0
for ch in r.iter_content(1024*1024):
    chunks.append(ch); dl+=len(ch)
    if tot: print(f'\r  {dl/1e6:.0f}/{tot/1e6:.0f} MB',end='')
print('\nExtracting...')
zipfile.ZipFile(io.BytesIO(b''.join(chunks))).extractall('/content/ncm/')
print('Done.')


In [ ]:
# Run DeltIEt on both datasets
files_nca=sorted(glob.glob('/content/nca/**/*.csv',recursive=True))
files_ncm=sorted(glob.glob('/content/ncm/**/*.csv',recursive=True))
print('='*55)
print('DeltIEt v4 — VALIDATION   w=15  h=40  n_bins=5  p90')
print('='*55)
df_nca,res_nca=run_dataset(files_nca,'Dataset 1 — NCA 18650')
df_ncm,res_ncm=run_dataset(files_ncm,'Dataset 2 — NCM 18650')
tk=df_nca['has_knee'].sum()+df_ncm['has_knee'].sum()
td=df_nca['det'].sum()+df_ncm['det'].sum()
al=pd.concat([df_nca[df_nca['det']]['lead'],df_ncm[df_ncm['det']]['lead']])
print()
print('='*55)
print('COMBINED SUMMARY')
print(f'Cells with knee: {tk}  |  Detected: {td}  ({100*td/tk:.1f}%)')
print(f'Lead  median={al.median():.0f}  mean={al.mean():.0f}  min={al.min()}  max={al.max()}')
print('='*55)


In [ ]:
# ── False-Positive Analysis ──────────────────────────────────────────────────
#
# Two complementary metrics:
#
#  1. CELL-LEVEL FP  — alarm fires in a cell that never reached 80% (no knee).
#     These are the only genuine false positives: the battery was healthy
#     throughout its recorded life and the algorithm still raised an alarm.
#
#  2. EVENT-LEVEL (alarm count per cell) — for each correctly detected cell,
#     how many individual alarm events fired before vs. after the knee.
#     Events before the knee are early-warning signals (desired);
#     events after the knee are late/redundant but not false positives.
#     The avg events/cell metric shows the alarm is not a spammer.
#
# ─────────────────────────────────────────────────────────────────────────────

def fp_analysis(df_s, results, label):
    rows = []
    for _, row in df_s.iterrows():
        name     = row['name']
        r, _     = results[name]
        alarms   = np.where(r['alarm'].values)[0]
        n_alarms = int(len(alarms))
        has_knee = bool(row['has_knee'])
        knee     = int(row['knee']) if has_knee else None

        if not has_knee:
            # No knee: every alarm is a cell-level false positive
            n_pre  = n_alarms
            n_post = 0
            fp_cell = int(n_alarms > 0)
        else:
            n_pre   = int(np.sum(alarms < knee))
            n_post  = int(np.sum(alarms >= knee))
            fp_cell = 0  # correctly detected

        rows.append(dict(
            Cell      = name,
            Knee      = knee if has_knee else '—',
            has_knee  = has_knee,
            N_alarms  = n_alarms,
            Before    = n_pre,
            After     = n_post,
            FP_cell   = fp_cell,
            Detected  = bool(row['det']),
        ))

    df = pd.DataFrame(rows)
    no_knee  = int((~df['has_knee']).sum())
    fp_cells = int(df['FP_cell'].sum())
    det      = df[df['Detected']]
    ta       = int(det['N_alarms'].sum())
    bp       = int(det['Before'].sum())
    ap       = int(det['After'].sum())

    SEP = '=' * 60
    print(f'\n{SEP}')
    print(f'  FALSE-POSITIVE ANALYSIS — {label}')
    print(SEP)
    print(f'  Total cells : {len(df)}')
    print(f'    with knee (degraded to 80%)  : {int(df.has_knee.sum())}')
    print(f'    without knee (stayed healthy): {no_knee}')
    if no_knee > 0:
        print(f'\n  Cell-level FP (alarm in healthy cell): '
              f'{fp_cells} / {no_knee} = {100*fp_cells/no_knee:.1f}%')
    else:
        print(f'\n  Cell-level FP: {fp_cells}  '
              f'(all cells reached the knee — 0 healthy cells to test against)')
    if len(det) > 0:
        print(f'\n  Event-level — detected cells (n={len(det)}):')
        print(f'    Total alarm events  : {ta}')
        print(f'    Before knee (early) : {bp}  ({100*bp/ta:.1f}%)  <- early-warning signals')
        print(f'    After  knee (late)  : {ap}  ({100*ap/ta:.1f}%)')
        print(f'    Avg events / cell   : {ta/len(det):.1f}')
    print()

    # Per-cell table (display columns only)
    tbl = df[['Cell','Knee','N_alarms','Before','After','FP_cell']].copy()
    tbl.columns = ['Cell','Knee_cycle','Alarms_total','Before_knee','After_knee','FP_cell']
    print(tbl.to_string(index=False))
    return df


df_fp_nca = fp_analysis(df_nca, res_nca, 'NCA 18650 — Dataset 1')
df_fp_ncm = fp_analysis(df_ncm, res_ncm, 'NCM 18650 — Dataset 2')

# ── Combined summary ─────────────────────────────────────────────────────────
all_fp   = pd.concat([df_fp_nca, df_fp_ncm], ignore_index=True)
no_knee  = int((~all_fp['has_knee']).sum())
fp_total = int(all_fp['FP_cell'].sum())
det_all  = all_fp[all_fp['Detected']]
ta = int(det_all['N_alarms'].sum())
bp = int(det_all['Before'].sum())

SEP = '=' * 60
print(f'\n{SEP}')
print('  COMBINED FALSE-POSITIVE SUMMARY  (NCA + NCM)')
print(SEP)
if no_knee > 0:
    print(f'  Cell-level FP rate : {fp_total} / {no_knee} = {100*fp_total/no_knee:.1f}%')
else:
    print(f'  Cell-level FP rate : 0  (no healthy cells in either dataset)')
if ta > 0:
    print(f'  Event-level before knee : {bp} / {ta} = {100*bp/ta:.1f}%  (early-warning)')
    print(f'  Event-level after  knee : {ta-bp} / {ta} = {100*(ta-bp)/ta:.1f}%  (late/redundant)')
    print(f'  Avg alarm events / cell : {ta/len(det_all):.1f}')
print(SEP)


In [ ]:
# Plot 4 best cells per dataset
def plot_best(df_s, results, label, n=4):
    data=[(row['name'],*results[row['name']],int(row['knee']),int(row['fa']),int(row['lead']))
          for _,row in df_s[df_s['det']].iterrows()]
    data.sort(key=lambda x:-x[5]); data=data[:n]
    fig=plt.figure(figsize=(16,4.5*len(data)))
    gs=gridspec.GridSpec(len(data),1,hspace=0.55)
    for i,(name,r,cap_arr,knee,fa,lead) in enumerate(data):
        ax1=fig.add_subplot(gs[i]); ax2=ax1.twinx()
        cy=np.arange(len(r)); cap=r['cap'].values
        di=np.where(np.isnan(r['DeltIEt'].values),0,r['DeltIEt'].values)
        thr=r['threshold'].values
        ax1.plot(cy,cap,color='#185FA5',lw=1.8,label='Capacity')
        ax1.axvline(knee,color='#E24B4A',lw=1.5,ls='--',label=f'Knee 80% (c.{knee})')
        ax1.axvline(fa,color='#EF9F27',lw=1.5,ls='--',label=f'1st alarm (c.{fa})')
        ax1.axhline(0.8*cap[0],color='#E24B4A',lw=0.8,ls=':',alpha=0.5)
        ym,yM=cap.min()*0.94,cap.max()*1.03
        ax1.fill_betweenx([ym,yM],fa,knee,alpha=0.10,color='#EF9F27')
        ax1.set_ylim(ym,yM)
        ax2.plot(cy,di,color='#639922',lw=0.9,alpha=0.75)
        ax2.plot(cy,np.where(np.isnan(thr),np.nan,thr),color='#639922',lw=0.8,ls=':',alpha=0.5)
        am=r['alarm'].values
        ax1.scatter(cy[am],cap[am],color='#EF9F27',s=22,zorder=5,label='Alarm')
        ax1.set_title(f'{name}  —  Lead: {lead} cycles  |  Rolling p90',fontsize=11)
        ax1.set_xlabel('Cycle'); ax1.set_ylabel('Capacity',color='#185FA5')
        ax2.set_ylabel('DeltIEt',color='#639922')
        lines,labs=ax1.get_legend_handles_labels()
        ax1.legend(lines,labs,loc='upper right',fontsize=8)
    plt.suptitle(f'DeltIEt v4 — {label}\nZenodo 10.5281/zenodo.6405084 · No training · No look-ahead',
                 fontsize=12,y=1.02)
    plt.tight_layout(); plt.show()

plot_best(df_nca,res_nca,'NCA 18650 — Dataset 1')
plot_best(df_ncm,res_ncm,'NCM 18650 — Dataset 2')


---
## Summary

| Dataset | Chemistry | Cells | Detection rate | Median lead |
|---|---|---|---|---|
| Zenodo Dataset 1 | NCA 18650 | 42 | **100%** (42/42) | 256 cycles |
| Zenodo Dataset 2 | NCM 18650 | 24 | **100%** (24/24) | 551 cycles |

Same parameters across both chemistries. No retraining. No look-ahead.

### False-Positive Rate

- **Cell-level FP** (alarm in a cell that never reached 80% degradation): see output of the False-Positive Analysis cell above.
- **Event-level**: all alarm events in correctly-detected cells that fired *before* the knee are early-warning signals — they are the mechanism of detection, not false positives.

**Patent:** BR 10 2026 012158 4 — INPI Brazil — filed 18 May 2026  
**Contact:** deltiet.research@gmail.com
